# From Failure to Fix: MAST classification — project Steps 1–6

This notebook contains the six project steps through classifier implementation. Step 6 contains its eight technical substeps, numbered 6.1–6.8. Historical experiment and recovery notebooks are archived in `backups`; they are not additional project tasks.

**Run from the beginning:** select the project `.venv` Python kernel, restart it, and Run All with `RUN_NEW_GEMINI_BATCH=False`. Public datasets download only when missing. Local checks and saved results are reproduced without new Gemini classifications. To deliberately classify new traces, use the single bounded-batch control under **6.8**, only when no other evaluation is active. Never share `.env`.

**Status is evidence-based:** successful code execution is not proof every model label is correct. The sample classifier has live pilot results with failures preserved. Full-corpus evaluation is a separate extension and remains incomplete while traces are unattempted.

Team handoff: [Teammate guide: Steps 1–6 and safe continuation](TEAMMATE_GUIDE_STEPS_1_TO_6.md).


## Setup — verified public dataset download

The first code cell downloads the pinned public full and human-labelled datasets only if missing (about 202 MB total), checks their hashes, and restores required local manifests. No Gemini key is needed for setup. Live classification uses Gemini, not the legacy Groq provider. A working Python kernel is required; `python-dotenv` is needed for live API execution.


In [1]:
# FIRST CODE CELL: download pinned public datasets only when missing.
# Fresh machine: approximately 202 MB. No Gemini key or pip packages required here.
from pathlib import Path
import importlib
import gemini_workflow as workflows
importlib.reload(workflows)
ROOT = Path.cwd()
if not (ROOT / "gemini_workflow.py").exists():
    raise RuntimeError("Open the notebook from the project folder containing gemini_workflow.py.")
DOWNLOAD_DATASETS = True  # Public Hugging Face download only; NOT Gemini execution.
ROOT = workflows.setup_project(ROOT, download=DOWNLOAD_DATASETS)
workflows.set_api_enabled(False)


Downloaded and verified: MAD_human_labelled_dataset.json
Downloaded and verified: MAD_full_dataset.json
Prepared pinned human records: 19
Setup complete. Public data and frozen support files verified. No Gemini requests.
Saved Gemini results are optional local artifacts; no saved results are downloaded or recreated.


## Step 1 — Choose the MAST failure-category scope

The project initially considered a smaller category focus; the user later requested all 14 for an exploratory classifier. This does not establish which categories are most common or select the final 2–3 remediation targets. The four-category baseline is preserved historically.


In [2]:
import json, hashlib
workflow_v3 = workflows.get_mode("evidence_v3")
human_mode = workflows.get_mode("fourteen_category")
print("Current classifier: 14-category exploratory Gemini judge")
print("Categories:")
for category in workflow_v3.DEFINITIONS:
    print(" -", category)
print("Final remediation focus is a later team decision; predicted frequency is not ground truth.")


Current classifier: 14-category exploratory Gemini judge
Categories:
 - disobey_task_specification
 - disobey_role_specification
 - step_repetition
 - loss_of_conversation_history
 - unaware_of_termination_conditions
 - conversation_reset
 - fail_to_ask_for_clarification
 - task_derailment
 - information_withholding
 - ignored_other_agents_input
 - reasoning_action_mismatch
 - premature_termination
 - no_or_incomplete_verification
 - incorrect_verification
Final remediation focus is a later team decision; predicted frequency is not ground truth.


## Step 2 — Review the paper and taxonomy definitions

Use the MAST paper and source definitions to keep meaning consistent. Definitions are displayed below; source category headings/codes and annotation rounds are retained separately. Numeric codes changed meanings between revisions and are not universal identifiers. Displaying definitions does not certify that every teammate has read the paper or agreed on mappings. Paper: https://arxiv.org/abs/2503.13657


In [3]:
for category, definition in workflow_v3.DEFINITIONS.items():
    print(category + ": " + definition)
print("Source definitions are preserved; category equivalence is checked by definition body, not numeric code alone.")


disobey_task_specification: Failure to adhere to the specified constraints or requirements of a given task, leading to suboptimal or incorrect outcomes.
disobey_role_specification: Failure to adhere to the defined responsibilities and constraints of an assigned role, potentially leading to an agent behaving like another.
step_repetition: Unnecessary reiteration of previously completed steps in a process, potentially causing delays or errors in task completion.
loss_of_conversation_history: Unexpected context truncation, disregarding recent interaction history and reverting to an antecedent conversational state.
unaware_of_termination_conditions: Lack of recognition or understanding of the criteria that should trigger the termination of the agents’ interaction, potentially leading to unnecessary continuation.
conversation_reset: Unexpected or unwarranted restarting of a dialogue, potentially losing context and progress made in the interaction.
fail_to_ask_for_clarification: (between age

## Step 3 — Download and verify the MAST datasets

The setup cell performs the actual pinned download. The large corpus has AI-generated reference labels; the separate smaller dataset contains the three human annotators. Do not call the large corpus human-labelled ground truth.


In [4]:
REVISION = "95118ac951421753cf1deb87ddea3b01e693c41b"
RAW_DIR = ROOT / "data/raw" / ("mast_" + REVISION)
split_manifest = json.loads((ROOT / "data/processed/full_corpus_step6_95118ac95142/split_manifest.json").read_text(encoding="utf-8"))
human_records = json.loads((RAW_DIR / "MAD_human_labelled_dataset.json").read_text(encoding="utf-8"), parse_constant=lambda value: None)
print("Pinned upstream revision:", REVISION)
print("Full dataset bytes:", (RAW_DIR / "MAD_full_dataset.json").stat().st_size)
print("Full-corpus source records:", len({i for row in split_manifest["traces"] for i in row["source_record_indices"]}))
print("Unique full-corpus traces:", len(split_manifest["traces"]))
print("Human dataset records:", len(human_records))
print("Raw data hashes were verified by setup; source files are not edited.")


Pinned upstream revision: 95118ac951421753cf1deb87ddea3b01e693c41b
Full dataset bytes: 199574367
Full-corpus source records: 1642
Unique full-corpus traces: 1537
Human dataset records: 19
Raw data hashes were verified by setup; source files are not edited.


## Step 4 — Define the common trace-logging template

Use stable trace identity and preserve provenance, model output, review notes and uncertainty. `label_correct` stays unknown until an eligible human comparison/review; it is not set from an AI success claim. API records and source annotations already supply the underlying fields. The joined comparison and accuracy report belong to Step 7.


In [5]:
logging_template = {
    "trace_hash": None, "source_trace_id": None, "framework": None,
    "source_revision": REVISION, "annotation_round": None,
    "source_category_heading": None, "canonical_category": None,
    "human_votes": None, "majority_label": None, "model_prediction": None,
    "record_status": None, "label_correct": None, "review_notes": None,
}
print(json.dumps(logging_template, indent=2))


{
  "trace_hash": null,
  "source_trace_id": null,
  "framework": null,
  "source_revision": "95118ac951421753cf1deb87ddea3b01e693c41b",
  "annotation_round": null,
  "source_category_heading": null,
  "canonical_category": null,
  "human_votes": null,
  "majority_label": null,
  "model_prediction": null,
  "record_status": null,
  "label_correct": null,
  "review_notes": null
}


## Step 5 — Parse human annotations and calculate majority labels

Majority requires two agreeing Boolean votes. Missing/non-finite votes are missing, not False. Keep source schemes and original definitions; exact duplicate/conflicting references and unresolved category mappings are not silently combined. This cell shows the complete source-row count and the separately eligible mapped human references. It does not replace AI references with human labels.


In [6]:
source_ground_truth = []
for index, record in enumerate(human_records):
    for annotation in record["annotations"]:
        votes = [annotation.get("annotator_" + str(i)) for i in (1, 2, 3)]
        yes = sum(v is True for v in votes)
        no = sum(v is False for v in votes)
        majority = True if yes >= 2 else False if no >= 2 else None
        source_ground_truth.append({"record_index": index, "round": record.get("round"), "trace_hash": hashlib.sha256(record["trace"].encode("utf-8")).hexdigest(), "category_heading": annotation["failure mode"].splitlines()[0], "votes": [v if type(v) is bool else None for v in votes], "majority_label": majority})
human_data = human_mode.prepare(ROOT)
print("All original trace/category annotation rows:", len(source_ground_truth))
print("Mapped annotation rows retained:", len(human_data["annotations"]))
print("Eligible canonical human references:", len(human_data["references"]))
print("Conflicting annotation rows outside scoring:", len(human_data["conflicts"]))
print("Distinct unresolved source headings:", len(human_data["unmapped_definitions"]))
print("Example majority rows (no trace text):")
for row in source_ground_truth[:3]:
    print(json.dumps(row))


All original trace/category annotation rows: 316
Mapped annotation rows retained: 206
Eligible canonical human references: 190
Conflicting annotation rows outside scoring: 12
Distinct unresolved source headings: 15
Example majority rows (no trace text):
{"record_index": 0, "round": "Round 1", "trace_hash": "a6abc5b76eed3380e2c2bb906437e47a97ee6494edd8d6942e9ac7afdc5e8abf", "category_heading": "1.1 Poor task constraint compliance", "votes": [false, false, false], "majority_label": false}
{"record_index": 0, "round": "Round 1", "trace_hash": "a6abc5b76eed3380e2c2bb906437e47a97ee6494edd8d6942e9ac7afdc5e8abf", "category_heading": "1.2 Inconsistency between reasoning and action", "votes": [false, false, false], "majority_label": false}
{"record_index": 0, "round": "Round 1", "trace_hash": "a6abc5b76eed3380e2c2bb906437e47a97ee6494edd8d6942e9ac7afdc5e8abf", "category_heading": "1.3 Undetected conversation ambiguities and contradictions", "votes": [false, false, false], "majority_label": false

## Step 6 — Build an LLM-based classifier

**Deliverable:** versioned Gemini classifier, prompt/settings, strict parser tests and saved predictions with visible failure statuses. The eight substeps below implement this workflow. A live pilot exists; its model errors and scoped-review limitations remain documented. Full-dataset evaluation is optional to the original sample-classifier goal, but is being tracked separately because the user requested broad coverage. It is not complete.


### 6.1 Confirm scope and taxonomy

Use the selected canonical definitions while preserving original source headings/codes separately. The taxonomy is not changed to make predictions agree with references.


In [7]:
assert len(workflow_v3.DEFINITIONS) == 14
print("Scope: 14 categories; provider Gemini; model gemini-3.5-flash-lite")
print("Source annotation schemes are preserved. Unresolved mappings stay outside primary comparisons.")


Scope: 14 categories; provider Gemini; model gemini-3.5-flash-lite
Source annotation schemes are preserved. Unresolved mappings stay outside primary comparisons.


### 6.2 Create development and held-out samples

Prompt revisions use development only. Exact trace duplicates and conservative task components cannot cross the split. Near-duplicate absence is not proven; earlier human-evaluated traces remain exploratory.


In [8]:
rows = split_manifest["traces"]
dev_hashes = {r["trace_hash"] for r in rows if r["split"] == "development"}
test_hashes = {r["trace_hash"] for r in rows if r["split"] == "heldout"}
assert not dev_hashes & test_hashes
assert not {r["group_id"] for r in rows if r["split"] == "development"} & {r["group_id"] for r in rows if r["split"] == "heldout"}
print("Development:", len(dev_hashes), "| Held-out:", len(test_hashes))
print("Exact trace overlap: 0 | conservative component overlap: 0")


Development: 1214 | Held-out: 323
Exact trace overlap: 0 | conservative component overlap: 0


### 6.3 Repair and version the prompt

Evidence-v3 contains explicit definitions, label meanings, evidence requirements and a trace boundary. Recorded instructions are data, not commands. No malformed numeric-code example is used. The prompt and classifier configuration remain frozen during broad evaluation; no held-out tuning is allowed.


In [9]:
prompt_check = workflow_v3.prompt_for("SYNTHETIC FULL TRACE")
assert "BEGIN RECORDED TRACE" in prompt_check and "END RECORDED TRACE" in prompt_check
assert "SYNTHETIC FULL TRACE" in prompt_check
print("Prompt version: full-corpus-fourteen-evidence-v3")
print("Frozen classifier hash:", workflow_v3.RECOVERY_CODE_SHA256)
print("Trace-boundary and full-input prompt checks passed.")


Prompt version: full-corpus-fourteen-evidence-v3
Frozen classifier hash: 0b2f1743d816f8da8a134e7dffbe1d94220e8fe922b5fe963ea3653b657b11ab
Trace-boundary and full-input prompt checks passed.


### 6.4 Preserve input fidelity

Live Gemini countTokens checks the full prompt before generation. Oversized inputs are blocked intact, not trimmed or silently dropped. Character-based estimates are not exact provider token counts.


In [10]:
progress = workflows.evaluation_progress(ROOT)
fidelity_checked = 0
for item in progress["folders"]:
    for path in (ROOT / item["folder"]).glob("*.json"):
        if len(path.stem) != 64:
            continue
        record = json.loads(path.read_text(encoding="utf-8"))
        assert record["trace_was_trimmed"] is False
        assert record["original_characters"] == record["sent_characters"]
        assert isinstance(record["counted_input_tokens"], int)
        fidelity_checked += 1
print("Saved full-input/token-count records checked:", fidelity_checked)
print("No new API calls were made by this check.")


Saved full-input/token-count records checked: 0
No new API calls were made by this check.


### 6.5 Request structured output

V3 documents a three-way adaptation: present/absent/insufficient_evidence, parsed to Boolean True/False/None. None is not a negative label. Each canonical category requires an explanation and quotes; positive findings require literal evidence. Existence of a quote does not prove the interpretation is correct.


In [11]:
schema = workflow_v3.schema()
assert len(schema["required"]) == 14
print("14 required category identifiers; fields: decision, evidence, quotes")
print("Stored labels: present=True; absent=False; insufficient_evidence=None")
print("Invalid or missing responses are unavailable, never automatically False.")


14 required category identifiers; fields: decision, evidence, quotes
Stored labels: present=True; absent=False; insufficient_evidence=None
Invalid or missing responses are unavailable, never automatically False.


### 6.6 Replace and test the permissive parser

Strict checks reject missing/extra/duplicate categories, duplicate JSON fields, wrong decision types, unsupported positives and non-verbatim quotes. Numeric identifiers are not mapped without their source meaning. Historical free-form 2.5/3.3 outputs were rejected as incompatible, not silently relabelled. Run the full tests with the project .venv.


In [12]:
parser_checks = workflows.parser_contract_checks()
print("Valid complete answer accepted:", parser_checks["valid_full_answer_passed"])
print("Invalid parser cases rejected:", len(parser_checks["invalid_cases_rejected"]))
print("Abstentions preserved:", parser_checks["abstention_preserved_not_false"])
replay = ROOT / "data/processed/step6_consolidated_audit/parser_replay_report.json"
if replay.exists():
    replay_report = json.loads(replay.read_text(encoding="utf-8"))
    print("Actual archived coded answers rejected:", len(replay_report["historical_coded_responses"]))
    print("Canonical baseline responses reparsed identically:", len(replay_report["canonical_saved_responses"]))


Valid complete answer accepted: True
Invalid parser cases rejected: 10
Abstentions preserved: True


### 6.7 Save resumable records

Records/manifests retain trace and prompt hashes, data revision/source scheme, provider/model/settings, full-input status, raw response/usage when returned, predictions and errors. Prior experiments remain intact. Any attempted v3 hash, including errors or unknown outcomes, is excluded from automatic resend.


In [13]:
progress = workflows.evaluation_progress(ROOT)
print("Preserved v3 experiment folders:", len(progress["folders"]))
print("Saved trace records:", sum(x["records"] for x in progress["folders"]))
print("No automatic retry of valid, invalid, failed, blocked or unknown-outcome trace records.")


Preserved v3 experiment folders: 0
Saved trace records: 0
No automatic retry of valid, invalid, failed, blocked or unknown-outcome trace records.


### 6.8 Pilot, freeze, and run a bounded Gemini batch

The 14-case online development pilot has 12 valid responses, one invalid quote response and one request failure without a saved response. Scoped AI-assisted evidence review and tests are documented; no exhaustive human correctness sign-off is fabricated. Known model errors are retained. Full held-out execution is blocked until development attempts are accounted for.

**Default:** RUN_NEW_GEMINI_BATCH=False displays current saved results only. **Live use:** set it True to attempt at most 20 untouched development traces / one million counted input tokens, using your .env key. Do not launch it while another run lock exists. HTTP 503 is a provider error; the batch stops, preserves that request, and does not retry it. The summary below refreshes AFTER a live batch, so it includes that batch rather than old pre-run totals.


In [42]:
import time

MAX_BATCHES_THIS_SESSION = 25
last_batch = None

try:
    for batch_number in range(1, MAX_BATCHES_THIS_SESSION + 1):
        progress = workflows.evaluation_progress(ROOT)

        if progress["batch_lock_exists"] or progress["supervisor_lock_exists"]:
            print("STOP: another run lock exists.")
            break

        remaining = progress["coverage"]["development"]["statuses"].get(
            "unattempted", 0
        )
        if remaining == 0:
            print("All development traces have been attempted.")
            break

        print(
            f"\nBatch {batch_number}: "
            f"{remaining} development traces remain.",
            flush=True,
        )

        workflows.set_api_enabled(True)
        try:
            last_batch = workflows.run_step6_batch(
                ROOT,
                split="development",
                max_generations=20,
                max_input_tokens=1_000_000,
                allow_api=True,
                verbose=True,
            )
        finally:
            workflows.set_api_enabled(False)

        state = last_batch["state"]
        print("Batch ended:", state, flush=True)

        # Continue only after ordinary batch boundaries.
        if state not in {
            "batch_generation_cap_reached",
            "batch_input_budget_reached",
        }:
            print("Stopping:", last_batch.get("error", state))
            break

        made_progress = (
            last_batch.get("generation_requests", 0)
            + last_batch.get("blocked", 0)
        )
        if made_progress == 0:
            print("STOP: no progress; review the deferred trace.")
            break

        time.sleep(5)  # Existing runner also enforces request/token pacing.
    else:
        print("Session batch limit reached. Progress is saved.")

except KeyboardInterrupt:
    print("Interrupted. Saved attempts remain; review before resuming.")
except Exception as error:
    print("Stopped:", type(error).__name__, str(error))
finally:
    workflows.set_api_enabled(False)
    progress = workflows.print_evaluation_summary(
        ROOT, batch_result=last_batch
    )


Batch 1: 923 development traces remain.
3a6d3b13cb37 valid 2354 input tokens
3a74931f9a08 invalid_response 5285 input tokens
3a884e570547 valid 2502 input tokens
3a89c15b317e valid 2655 input tokens
3a8a455a57c6 valid 3315 input tokens
3aa5b221838a valid 2974 input tokens
3b56f7871fbb invalid_response 7763 input tokens
3ba06cdc874e valid 60861 input tokens
3be978d6f0dd valid 65559 input tokens
3d317f0907a9 valid 25050 input tokens
3d3ba04adfd8 valid 2230 input tokens
3d6e47d05847 valid 3722 input tokens
3d72efd01741 valid 4233 input tokens
3da2183350ff invalid_response 9009 input tokens
3dce7bf74d20 invalid_response 77777 input tokens
3de9a35c6348 valid 2954 input tokens
3e22e2e20f6b valid 4103 input tokens
3eb630229dd5 valid 58941 input tokens
3ec09d8fe0d7 valid 25615 input tokens
3ed4e57d2e37 valid 29333 input tokens
Batch ended: batch_generation_cap_reached

Batch 2: 903 development traces remain.
3edf0b9db917 invalid_response 3470 input tokens
3f1d4d22d37b valid 20472 input tokens

In [43]:
progress = workflows.print_evaluation_summary(ROOT)

CURRENT SAVED RESULTS (refreshed now)
Development: 1214 total | 349 valid | 728 unattempted
  Other outcomes (not negative labels): blocked_input_limit=3, invalid_response=131, request_error=3
Heldout: 323 total | 0 valid | 323 unattempted
Latest saved response/record: 2026-10-26T23:45:19.274268+00:00
No evaluation run lock is present.
Latest batch: paused_local_daily_request_cap
Batch checkpoint time: 2026-10-26T23:45:19.280126+00:00
  Generation attempts: 14 | valid: 9 | invalid: 5 | blocked: 0 | counted generation input tokens: 456,800
Full-dataset coverage complete: False
Validation is not correctness. Invalid, blocked and unknown outcomes are not scored as absent.


## Status and handoff to Step 7

The classifier, parser, versioned settings and sample predictions exist with documented errors. Invalid outputs are not repaired into successful labels. The full-dataset extension is **incomplete** while either split has unattempted traces; provider errors and blocks must stay in its final coverage report. Human manual pilot sign-off has not been recorded.

Step 7 is a separate comparison against eligible human labels and disagreement analysis. Agreement with upstream AI-reference labels is not human accuracy. Saved outputs and successful tests do not establish broad reliability.


## Step 7 — Compare predictions with reference labels
Partial results. Human-reference agreement and AI-reference agreement are reported separately.

In [37]:
import json
from pathlib import Path
from collections import Counter

step7_records = {}
step7_record_paths = {}

for folder in sorted(
    (ROOT / "saved_results").glob("gemini_full_corpus_evidence_v3_*")
):
    for path in sorted(folder.glob("*.json")):
        # Individual trace records use a 64-character hexadecimal hash.
        trace_hash = path.stem
        if len(trace_hash) != 64 or any(
            char not in "0123456789abcdef" for char in trace_hash
        ):
            continue

        record = json.loads(path.read_text(encoding="utf-8"))

        if record.get("trace_hash") != trace_hash:
            raise ValueError(f"Trace identity mismatch: {path.name}")
        if trace_hash in step7_records:
            raise ValueError(f"Duplicate trace record: {trace_hash}")

        step7_records[trace_hash] = record
        step7_record_paths[trace_hash] = str(path)

print("Saved trace records:", len(step7_records))
print("Statuses:")
for status, count in sorted(
    Counter(r.get("status", "unknown") for r in step7_records.values()).items()
):
    print(f"  {status}: {count}")

Saved trace records: 291
Statuses:
  blocked_input_limit: 2
  invalid_response: 70
  request_error: 3
  valid: 216


In [38]:
# Reuse Step 5's human-label preparation.
step7_human_data = human_mode.prepare(ROOT)

normalize = lambda text: " ".join(text.split())

# Match category meaning, not just category names or numeric codes.
if set(human_mode.DEFINITIONS) != set(workflow_v3.DEFINITIONS):
    raise ValueError("Human and classifier categories differ.")

for category in workflow_v3.DEFINITIONS:
    if normalize(human_mode.DEFINITIONS[category]) != normalize(
        workflow_v3.DEFINITIONS[category]
    ):
        raise ValueError(f"Definition mapping needs review: {category}")

step7_human_refs = {}

for row in step7_human_data["references"]:
    key = (row["trace_hash"], row["category"])
    if key in step7_human_refs:
        raise ValueError(f"Duplicate human reference: {key}")
    if type(row["label"]) is not bool:
        raise ValueError(f"Invalid human reference label: {key}")
    step7_human_refs[key] = row["label"]

print("Category-definition checks passed.")
print("Eligible human trace/category pairs:", len(step7_human_refs))
print(
    "Unique human-reference traces:",
    len({trace_hash for trace_hash, category in step7_human_refs}),
)
print("Conflicting annotation rows excluded:",
      len(step7_human_data["conflicts"]))
print("Unresolved source definitions:",
      len(step7_human_data["unmapped_definitions"]))

Category-definition checks passed.
Eligible human trace/category pairs: 190
Unique human-reference traces: 18
Conflicting annotation rows excluded: 12
Unresolved source definitions: 15


In [39]:
import pandas as pd

step7_overlap_rows = []

for (trace_hash, category), human_label in step7_human_refs.items():
    record = step7_records.get(trace_hash)

    step7_overlap_rows.append({
        "trace_hash": trace_hash,
        "category": category,
        "human_label": human_label,
        "saved_status": (
            record.get("status", "unknown")
            if record is not None
            else "no_saved_record"
        ),
    })

step7_overlap_df = pd.DataFrame(step7_overlap_rows)

print("Human-reference pairs by saved-result status:")
print(step7_overlap_df["saved_status"].value_counts().to_string())

valid_overlap = step7_overlap_df[
    step7_overlap_df["saved_status"] == "valid"
]

print("\nHuman-reference traces with valid saved responses:",
      valid_overlap["trace_hash"].nunique())
print("Pairs available for further validation:", len(valid_overlap))

Human-reference pairs by saved-result status:
saved_status
no_saved_record    180
valid               10

Human-reference traces with valid saved responses: 1
Pairs available for further validation: 10


In [28]:
import hashlib

step7_validated = {}
step7_validation_errors = {}

marker = "__STEP7_TRACE_MARKER__"
prefix, suffix = workflow_v3.prompt_for(marker).split(marker)

for trace_hash, record in step7_records.items():
    if record.get("status") != "valid":
        continue

    try:
        prompt = record["prompt"]

        if hashlib.sha256(prompt.encode("utf-8")).hexdigest() != record["prompt_hash"]:
            raise ValueError("Prompt hash mismatch")

        if not prompt.startswith(prefix) or not prompt.endswith(suffix):
            raise ValueError("Unexpected prompt template")

        end = len(prompt) - len(suffix) if suffix else len(prompt)
        trace_text = prompt[len(prefix):end]

        if hashlib.sha256(trace_text.encode("utf-8")).hexdigest() != trace_hash:
            raise ValueError("Trace content hash mismatch")

        if record.get("trace_was_trimmed") is not False:
            raise ValueError("Trace was trimmed or fidelity is unknown")

        candidates = record["raw_response"]["candidates"]
        if len(candidates) != 1 or candidates[0].get("finishReason") != "STOP":
            raise ValueError("Incomplete or unexpected response")

        response_text = "".join(
            part.get("text", "")
            for part in candidates[0].get("content", {}).get("parts", [])
            if not part.get("thought", False)
        )

        predictions = workflow_v3.parse_prediction(response_text)
        workflow_v3.validate_quotes(predictions, trace_text)

        if predictions != record.get("predictions"):
            raise ValueError("Saved predictions differ from reparsed response")

        step7_validated[trace_hash] = predictions

    except (ValueError, KeyError, TypeError, IndexError) as error:
        step7_validation_errors[trace_hash] = str(error)

print("Offline-validated trace responses:", len(step7_validated))
print("Additional validation failures:", len(step7_validation_errors))

for trace_hash, error in step7_validation_errors.items():
    print(trace_hash[:12], error)

Offline-validated trace responses: 184
Additional validation failures: 0


In [41]:
step7_human_rows = []

trace_metadata = {
    row["trace_hash"]: row
    for row in split_manifest["traces"]
}

for (trace_hash, category), human_label in step7_human_refs.items():
    record = step7_records.get(trace_hash)
    finding = step7_validated.get(trace_hash, {}).get(category, {})
    prediction = finding.get("present")

    if trace_hash not in trace_metadata:
        status = "outside_corpus_manifest"
    elif record is None:
        status = "unattempted"
    elif record.get("status") != "valid":
        status = record.get("status", "unknown")
    elif trace_hash not in step7_validated:
        status = "offline_validation_failed"
    elif type(prediction) is not bool:
        status = "insufficient_evidence"
    else:
        status = "comparable"

    step7_human_rows.append({
        "trace_hash": trace_hash,
        "framework": trace_metadata.get(trace_hash, {}).get("framework"),
        "split": trace_metadata.get(trace_hash, {}).get("split"),
        "category": category,
        "human_label": human_label,
        "gemini_label": prediction,
        "comparison_status": status,
        "match": human_label == prediction if status == "comparable" else None,
        "evidence": finding.get("evidence"),
        "quotes": finding.get("quotes"),
        "review_notes": "",
    })

step7_human_df = pd.DataFrame(step7_human_rows)

print("Comparison coverage:")
print(step7_human_df["comparison_status"].value_counts().to_string())

print("\nComparable human and Gemini labels:")
display(
    step7_human_df.loc[
        step7_human_df["comparison_status"] == "comparable",
        ["category", "human_label", "gemini_label", "match"],
    ]
)

Comparison coverage:
comparison_status
outside_corpus_manifest    160
unattempted                 20
comparable                  10

Comparable human and Gemini labels:


,category,human_label,gemini_label,match
0,conversation_reset,False,False,True
1,disobey_role_specification,False,False,True
2,disobey_task_specification,True,True,True
3,fail_to_ask_for_clarification,True,False,False
4,ignored_other_agents_input,False,False,True
5,information_withholding,False,False,True
6,loss_of_conversation_history,False,False,True
7,step_repetition,False,True,False
8,task_derailment,True,False,False
9,unaware_of_termination_conditions,False,False,True


In [30]:
def step7_metrics(rows):
    tp = fp = fn = tn = 0

    for row in rows:
        actual = bool(row["human_label"])
        predicted = bool(row["gemini_label"])

        if actual and predicted:
            tp += 1
        elif not actual and predicted:
            fp += 1
        elif actual and not predicted:
            fn += 1
        else:
            tn += 1

    n = tp + fp + fn + tn

    def divide(numerator, denominator):
        return numerator / denominator if denominator else None

    accuracy = divide(tp + tn, n)
    expected = (
        ((tp + fn) * (tp + fp) + (tn + fp) * (tn + fn)) / n**2
        if n else None
    )

    return {
        "n": n,
        "positive_human_labels": tp + fn,
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision": divide(tp, tp + fp),
        "recall": divide(tp, tp + fn),
        "F1": divide(2 * tp, 2 * tp + fp + fn),
        "accuracy": accuracy,
        "kappa": (
            (accuracy - expected) / (1 - expected)
            if expected is not None and expected != 1 else None
        ),
    }


comparable = step7_human_df[
    step7_human_df["comparison_status"] == "comparable"
]

metric_rows = []

for split in ("development", "heldout"):
    for category in workflow_v3.DEFINITIONS:
        selected = comparable[
            (comparable["split"] == split)
            & (comparable["category"] == category)
        ]
        metric_rows.append({
            "split": split,
            "category": category,
            **step7_metrics(selected.to_dict("records")),
        })

step7_human_metrics_df = pd.DataFrame(metric_rows)

display(step7_human_metrics_df)
print("NaN means undefined—not zero.")
print("Current human comparison covers",
      comparable["trace_hash"].nunique(), "unique trace(s).")

,split,category,n,positive_human_labels,TP,FP,FN,TN,precision,recall,F1,accuracy,kappa
0,development,disobey_task_specification,1,1,1,0,0,0,1.0,1.0,1.0,1.0,NaN
1,development,disobey_role_specification,1,0,0,0,0,1,NaN,NaN,NaN,1.0,NaN
2,development,step_repetition,1,0,0,1,0,0,0.0,NaN,0.0,0.0,0.0
3,development,loss_of_conversation_history,1,0,0,0,0,1,NaN,NaN,NaN,1.0,NaN
4,development,unaware_of_termination_conditions,1,0,0,0,0,1,NaN,NaN,NaN,1.0,NaN
5,development,conversation_reset,1,0,0,0,0,1,NaN,NaN,NaN,1.0,NaN
6,development,fail_to_ask_for_clarification,1,1,0,0,1,0,NaN,0.0,0.0,0.0,0.0
7,development,task_derailment,1,1,0,0,1,0,NaN,0.0,0.0,0.0,0.0
8,development,information_withholding,1,0,0,0,0,1,NaN,NaN,NaN,1.0,NaN
9,development,ignored_other_agents_input,1,0,0,0,0,1,NaN,NaN,NaN,1.0,NaN


NaN means undefined—not zero.
Current human comparison covers 1 unique trace(s).


In [31]:
step7_disagreements_df = step7_human_df[
    (step7_human_df["comparison_status"] == "comparable")
    & (step7_human_df["match"] == False)
].copy()

print("Human–Gemini disagreements:", len(step7_disagreements_df))

with pd.option_context("display.max_colwidth", 180):
    display(
        step7_disagreements_df[
            [
                "trace_hash",
                "category",
                "human_label",
                "gemini_label",
                "evidence",
                "quotes",
                "review_notes",
            ]
        ]
    )

Human–Gemini disagreements: 3


,trace_hash,category,human_label,gemini_label,evidence,quotes,review_notes
3,102983105332d602ea48b001f5dbabf33e696c60e79812b9e9a33f07fcc651cf,fail_to_ask_for_clarification,True,False,The agents successfully communicated and requested login credentials when needed.,[],
7,102983105332d602ea48b001f5dbabf33e696c60e79812b9e9a33f07fcc651cf,step_repetition,False,True,The agent repeated the action of listing all notes multiple times without new information or necessity.,"[send_message(app_name='simple_note', message='List all my notes'), send_message(app_name='simple_note', message='List all notes')]",
8,102983105332d602ea48b001f5dbabf33e696c60e79812b9e9a33f07fcc651cf,task_derailment,True,False,The agents stayed focused on interacting with the simple note application throughout the trace.,[],


In [32]:
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
step7_output = ROOT / "saved_results" / "step7" / timestamp
step7_output.mkdir(parents=True, exist_ok=False)

# Save tables. Undefined metrics are exported as empty cells.
step7_human_df.to_csv(
    step7_output / "human_comparison.csv", index=False, na_rep=""
)
step7_human_metrics_df.to_csv(
    step7_output / "human_metrics.csv", index=False, na_rep=""
)
step7_disagreements_df.to_csv(
    step7_output / "human_disagreements_for_review.csv",
    index=False,
    na_rep="",
)

# Coverage uses the same saved-record snapshot as the comparison.
coverage = {}
for split in ("development", "heldout"):
    hashes = {
        row["trace_hash"]
        for row in split_manifest["traces"]
        if row["split"] == split
    }
    coverage[split] = dict(Counter(
        step7_records[h].get("status", "unknown")
        if h in step7_records else "unattempted"
        for h in hashes
    ))

summary = {
    "created_at_utc": timestamp,
    "scope": "Partial exploratory human-reference comparison",
    "dataset_revision": split_manifest["dataset_revision"],
    "saved_records": len(step7_records),
    "offline_validated_records": len(step7_validated),
    "coverage": coverage,
    "human_comparison_statuses": {
        str(status): int(count)
        for status, count in
        step7_human_df["comparison_status"].value_counts().items()
    },
    "comparable_human_traces": int(comparable["trace_hash"].nunique()),
    "comparable_human_pairs": len(comparable),
    "disagreements": len(step7_disagreements_df),
    "conflicting_annotation_rows_excluded":
        len(step7_human_data["conflicts"]),
    "unresolved_source_definitions":
        len(step7_human_data["unmapped_definitions"]),
    "limitations": [
        "Only one matched human-reference trace in this snapshot.",
        "Undefined metrics are not zero.",
        "No held-out performance claim.",
        "Disagreements have not yet been independently reviewed.",
    ],
}

audit = {
    "annotations": step7_human_data["annotations"],
    "conflicts": step7_human_data["conflicts"],
    "unmapped_definitions": step7_human_data["unmapped_definitions"],
    "offline_validation_errors": step7_validation_errors,
    "record_paths": step7_record_paths,
}

for filename, content in [
    ("summary.json", summary),
    ("human_reference_audit.json", audit),
]:
    (step7_output / filename).write_text(
        json.dumps(content, indent=2, ensure_ascii=False, allow_nan=False),
        encoding="utf-8",
    )

print("Step 7 snapshot saved to:", step7_output)

Step 7 snapshot saved to: C:\Study\3rd sem\soft archi\mast\MAST_RADP\saved_results\step7\20261025T213509165290Z
